In [13]:
# ============================================================
# 1- Phone Number Normalization
# ============================================================

import re

_PERSIAN_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789"
)

_ARABIC_DIGITS = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩",
    "0123456789"
)


def normalize_iran_phone(value) -> str:
    """
    Return an Iranian mobile number in local 10-digit form.

    Example:
        9121234567

    Accepted formats:
        9121234567
        09121234567
        +989121234567
        00989121234567

    Persian and Arabic-Indic digits are also accepted.
    """

    if value is None:
        raise ValueError("Phone number is empty")

    # Normalize Unicode digits first.
    s = str(value).strip()
    s = s.translate(_PERSIAN_DIGITS)
    s = s.translate(_ARABIC_DIGITS)

    # Reject obvious Excel numeric artifacts.
    if re.fullmatch(r"\d+\.0+", s):
        raise ValueError(
            f"Phone number appears to have been converted to a numeric "
            f"Excel value: {value!r}"
        )

    # Keep digits and an optional leading '+'.
    s = re.sub(r"[^\d+]", "", s)

    # Convert international Iranian formats to local format.
    if s.startswith("+98"):
        s = s[3:]
    elif s.startswith("0098"):
        s = s[4:]
    elif s.startswith("98") and len(s) == 12:
        s = s[2:]
    elif s.startswith("0"):
        s = s[1:]

    # Final Iranian mobile validation.
    if not re.fullmatch(r"9\d{9}", s):
        raise ValueError(
            f"Invalid Iranian mobile number: {value!r}"
        )

    return s

In [22]:
# ============================================================
# 2- Import Excel Data into the DataFrame
# ============================================================

from pathlib import Path
import pandas as pd

EXCEL_FILE = Path(
    "/home/saeed/repositories/contacts/import_eitaa/contacts-501-1000.xlsx"
)

# ------------------------------------------------------------
# 2.1 - Validate the Excel file
# ------------------------------------------------------------

if not EXCEL_FILE.is_file():
    raise FileNotFoundError(
        f"Excel file not found: {EXCEL_FILE}"
    )

# ------------------------------------------------------------
# 2.2 - Read Excel data
# ------------------------------------------------------------

df = pd.read_excel(
    EXCEL_FILE,
    header=None,
    usecols=[0, 1],
    dtype=str,
    keep_default_na=False,
    engine="openpyxl",
)

df.columns = ["full_name", "phone"]

# ------------------------------------------------------------
# 2.3 - Clean the imported data
# ------------------------------------------------------------

df["full_name"] = df["full_name"].str.strip()
df["phone"] = df["phone"].str.strip()

# Remove completely empty rows.
df = df.loc[
    ~(df["full_name"].eq("") & df["phone"].eq(""))
].copy()

df.reset_index(drop=True, inplace=True)

# ------------------------------------------------------------
# 2.4 - Normalize and validate phone numbers
# ------------------------------------------------------------

normalized_phones = []
validation_errors = []

for phone in df["phone"]:
    try:
        normalized_phone = normalize_iran_phone(phone)
        normalized_phones.append(normalized_phone)
        validation_errors.append(None)

    except ValueError as exc:
        normalized_phones.append(None)
        validation_errors.append(str(exc))

df["phone"] = normalized_phones
df["validation_error"] = validation_errors

# ------------------------------------------------------------
# 2.5 - Separate valid and invalid contact records
# ------------------------------------------------------------

invalid_contacts = df.loc[
    df["validation_error"].notna()
].copy()

valid_contacts = df.loc[
    df["validation_error"].isna()
].copy()

# ------------------------------------------------------------
# 2.6 - Identify duplicate phone numbers
# ------------------------------------------------------------

valid_contacts["is_duplicate"] = valid_contacts[
    "phone"
].duplicated(keep="first")

# ------------------------------------------------------------
# 2.7 - Assign contact IDs
# ------------------------------------------------------------

valid_contacts = valid_contacts.reset_index(drop=True)

valid_contacts.insert(
    0,
    "contact_id",
    range(len(valid_contacts))
)

# ------------------------------------------------------------
# 2.8 - Generate the final DataFrame
# ------------------------------------------------------------

contacts_df = valid_contacts[
    ["contact_id", "full_name", "phone"]
].copy()

# Make df the final DataFrame with exactly three columns.
df = contacts_df.copy()

# ------------------------------------------------------------
# 2.9 - Display the import summary
# ------------------------------------------------------------

print("=" * 60)
print("EXCEL CONTACT IMPORT SUMMARY")
print("=" * 60)

print(f"Excel file:             {EXCEL_FILE.name}")
print(f"Imported records:       {len(invalid_contacts) + len(valid_contacts)}")
print(f"Valid contacts:         {len(contacts_df)}")
print(f"Invalid records:        {len(invalid_contacts)}")
print(
    "Duplicate phone rows:   "
    f"{int(valid_contacts['is_duplicate'].sum())}"
)

# ------------------------------------------------------------
# 2.10 - Display invalid records for review
# ------------------------------------------------------------

if not invalid_contacts.empty:
    print("\\nInvalid records requiring review:")

    display(
        invalid_contacts[
            ["full_name", "phone", "validation_error"]
        ]
    )

# ------------------------------------------------------------
# 2.11 - Optional: display duplicate contacts
# ------------------------------------------------------------

duplicates_df = valid_contacts.loc[
    valid_contacts["is_duplicate"]
].copy()

if not duplicates_df.empty:
    print("\\nDuplicate phone numbers requiring review:")
    display(duplicates_df)

EXCEL CONTACT IMPORT SUMMARY
Excel file:             contacts-501-1000.xlsx
Imported records:       500
Valid contacts:         500
Invalid records:        0
Duplicate phone rows:   0


In [ ]:
df

In [5]:
# ============================================================
# 3- Selenium Chrome session persistence
# ============================================================
# Clean, isolated Selenium Chrome session

from pathlib import Path
from tempfile import mkdtemp
from typing import Optional
import shutil

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.webdriver import WebDriver

# ChromeDriver

CHROMEDRIVER = (
        Path.cwd()
        / "drivers"
        / "chromedriver"
        / "chromedriver"
)

# Create a completely new temporary Chrome User Data Directory

TEMP_CHROME_PROFILE = Path(
    mkdtemp(prefix="selenium-chrome-")
)

# Close previous Selenium session, if one exists

previous_driver: Optional[WebDriver] = globals().get("driver")

if previous_driver is not None:
    try:
        previous_driver.quit()
        print("\nPrevious Selenium driver closed.")
    except Exception as e:
        print("\nPrevious Selenium driver could not be closed:")
        print(repr(e))

# Chrome options

options = Options()

options.add_argument(
    f"--user-data-dir={TEMP_CHROME_PROFILE}"
)

options.add_argument("--start-maximized")

#  Explicit manual ChromeDriver
#  Selenium Manager is not used for driver discovery.


service = Service(
    executable_path=str(CHROMEDRIVER)
)

# Create new isolated Chrome session


driver: WebDriver = webdriver.Chrome(
    service=service,
    options=options,
)

driver.set_page_load_timeout(60)


In [20]:
# ============================================================
# 4- Close Selenium and remove temporary Chrome profile
# ============================================================

if "driver" in globals() and driver is not None:
    try:
        driver.quit()
        print("Selenium Chrome session closed.")
    finally:
        driver = None

if "TEMP_CHROME_PROFILE" in globals():
    shutil.rmtree(
        TEMP_CHROME_PROFILE,
        ignore_errors=True
    )

    print(
        "Temporary Selenium Chrome profile removed."
    )

Selenium Chrome session closed.
Temporary Selenium Chrome profile removed.


In [7]:
# ============================================================
# Extract existing Eitaa contacts into df_exist
# ============================================================
import time
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

SCROLL_PAUSE = 0.8
STABLE_BOTTOM_LIMIT = 4
MAX_SCROLL_STEPS = 300

contacts = {}  # peer_id -> full_name


def get_contacts_list():
    return driver.find_element(By.CSS_SELECTOR, "ul#contacts")


def get_scroll_metrics():
    """Find the actual scrollable ancestor of the contacts list."""
    contacts_list = get_contacts_list()

    result = driver.execute_script("""
        const list = arguments[0];
        let el = list.parentElement;

        while (el) {
            const style = getComputedStyle(el);

            if (
                (style.overflowY === "auto" ||
                 style.overflowY === "scroll") &&
                el.scrollHeight > el.clientHeight
            ) {
                return {
                    scrollTop: el.scrollTop,
                    clientHeight: el.clientHeight,
                    scrollHeight: el.scrollHeight
                };
            }

            el = el.parentElement;
        }

        return null;
    """, contacts_list)

    if result is None:
        raise RuntimeError("Could not find the scrollable contacts container.")

    return result


def collect_visible_contacts():
    """Collect currently rendered contacts without clicking them."""
    rows = driver.find_elements(
        By.CSS_SELECTOR, "ul#contacts > a.chatlist-chat"
    )

    added = 0

    for row in rows:
        peer_id = row.get_attribute("data-peer-id")

        name_elements = row.find_elements(
            By.CSS_SELECTOR, "span.peer-title"
        )

        if not peer_id or not name_elements:
            continue

        full_name = name_elements[0].text.strip()

        if full_name and peer_id not in contacts:
            contacts[peer_id] = full_name
            added += 1

    return added


# Wait until the contact list is available.
WebDriverWait(driver, 15).until(
    lambda d: len(d.find_elements(
        By.CSS_SELECTOR, "ul#contacts > a.chatlist-chat"
    )) > 0
)

# Start at the top to avoid missing contacts above the current position.
contacts_list = get_contacts_list()

driver.execute_script("""
    const list = arguments[0];
    let el = list.parentElement;

    while (el) {
        const style = getComputedStyle(el);

        if (
            (style.overflowY === "auto" ||
             style.overflowY === "scroll") &&
            el.scrollHeight > el.clientHeight
        ) {
            el.scrollTop = 0;
            return;
        }

        el = el.parentElement;
    }
""", contacts_list)

time.sleep(SCROLL_PAUSE)
collect_visible_contacts()

stable_bottom_steps = 0

for step in range(1, MAX_SCROLL_STEPS + 1):
    before = get_scroll_metrics()

    # Scroll down within the actual contacts container.
    contacts_list = get_contacts_list()

    driver.execute_script("""
        const list = arguments[0];
        let el = list.parentElement;

        while (el) {
            const style = getComputedStyle(el);

            if (
                (style.overflowY === "auto" ||
                 style.overflowY === "scroll") &&
                el.scrollHeight > el.clientHeight
            ) {
                const distance = Math.max(
                    300, Math.floor(el.clientHeight * 0.75)
                );

                el.scrollTop = Math.min(
                    el.scrollTop + distance,
                    el.scrollHeight
                );

                return;
            }

            el = el.parentElement;
        }
    """, contacts_list)

    time.sleep(SCROLL_PAUSE)

    added = collect_visible_contacts()
    after = get_scroll_metrics()

    at_bottom = (
            after["scrollTop"] + after["clientHeight"]
            >= after["scrollHeight"] - 3
    )

    no_growth = (
            after["scrollHeight"] <= before["scrollHeight"]
    )

    if at_bottom and no_growth and added == 0:
        stable_bottom_steps += 1
    else:
        stable_bottom_steps = 0

    print(
        f"Step {step}: "
        f"visible={len(driver.find_elements(By.CSS_SELECTOR, 'ul#contacts > a.chatlist-chat'))}, "
        f"new={added}, "
        f"unique_total={len(contacts)}, "
        f"scrollTop={after['scrollTop']}, "
        f"scrollHeight={after['scrollHeight']}, "
        f"stable_bottom={stable_bottom_steps}/{STABLE_BOTTOM_LIMIT}"
    )

    if stable_bottom_steps >= STABLE_BOTTOM_LIMIT:
        print("\nReached a stable bottom; stopping extraction.")
        break
else:
    print(
        "\nWARNING: Maximum scroll steps reached. "
        "The list may not be fully loaded."
    )

# Build the final DataFrame in first-seen order.
df_exist = pd.DataFrame(
    [
        {"peer_id": peer_id, "full_name": full_name}
        for peer_id, full_name in contacts.items()
    ],
    columns=["peer_id", "full_name"]
)

print("\nExtraction summary")
print("------------------")
print("Unique contacts extracted:", len(df_exist))
print("Unique peer IDs:", df_exist["peer_id"].nunique())
print("DataFrame shape:", df_exist.shape)


Step 1: visible=40, new=20, unique_total=40, scrollTop=386, scrollHeight=2306, stable_bottom=0/4
Step 2: visible=40, new=0, unique_total=40, scrollTop=986, scrollHeight=2306, stable_bottom=0/4
Step 3: visible=60, new=20, unique_total=60, scrollTop=1506, scrollHeight=3426, stable_bottom=0/4
Step 4: visible=60, new=0, unique_total=60, scrollTop=2106, scrollHeight=3426, stable_bottom=0/4
Step 5: visible=80, new=20, unique_total=80, scrollTop=2626, scrollHeight=4546, stable_bottom=0/4
Step 6: visible=80, new=0, unique_total=80, scrollTop=3226, scrollHeight=4546, stable_bottom=0/4
Step 7: visible=100, new=20, unique_total=100, scrollTop=3746, scrollHeight=5666, stable_bottom=0/4
Step 8: visible=100, new=0, unique_total=100, scrollTop=4346, scrollHeight=5666, stable_bottom=0/4
Step 9: visible=120, new=20, unique_total=120, scrollTop=4866, scrollHeight=6786, stable_bottom=0/4
Step 10: visible=120, new=0, unique_total=120, scrollTop=5466, scrollHeight=6786, stable_bottom=0/4
Step 11: visible=1

In [20]:
# ============================================================
# df = df - df_exist
# ============================================================
import re
import unicodedata
import pandas as pd


def normalize_name(name):
    if pd.isna(name):
        return ""

    name = unicodedata.normalize("NFKC", str(name))

    # Standardize Arabic/Persian letter variants.
    name = name.translate(str.maketrans({
        "ي": "ی",
        "ى": "ی",
        "ك": "ک",
        "ئ": "ی",
        "ة": "ه",
        "ۀ": "ه",
    }))

    # Remove invisible formatting characters.
    name = "".join(
        ch for ch in name
        if unicodedata.category(ch) != "Cf"
    )

    # Standardize whitespace.
    name = re.sub(r"\s+", " ", name).strip()

    return name


existing_names = set(
    df_exist["full_name"].map(normalize_name)
) - {""}

df = df.loc[
    ~df["full_name"].map(normalize_name).isin(existing_names)
].copy()

print("Remaining rows:", len(df))



Remaining rows: 473


In [9]:
# ============================================================
# 5- Selenium Selectors
# ============================================================

from selenium.webdriver.common.by import By


class Selectors:
    # 1. "+" button inside Contacts
    ADD_CONTACT_BUTTON = (
        By.XPATH,
        "//button[contains(@class, 'btn-circle') "
        "and contains(@class, 'btn-corner') "
        "and .//span[@data-icon='add']]"
    )

    # 2. First name field
    FIRST_NAME = (
        By.CSS_SELECTOR,
        'div.input-field:first-child > div.input-field-input[contenteditable="true"]'
    )

    # 3. Phone number field
    PHONE = (
        By.CSS_SELECTOR,
        'div.input-field-phone > '
        'div.input-field-input[contenteditable="true"]'
    )

    # 4. Add/Save button
    SAVE_BUTTON = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-primary.btn-color-primary"
    )

    # 5. Toast shown when the phone number is not registered on Eitaa
    UNREGISTERED_TOAST = (
        By.CSS_SELECTOR,
        "div.toast.is-visible"
    )

    # 6. Close Add Contact window ( × button )
    CLOSE_CONTACT_WINDOW = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-icon.popup-close"
    )

In [ ]:
DELETE
ME!
# ============================================================
# TEST - Section 5: Selenium Selectors
# ============================================================

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

WAIT_TIMEOUT = 10

# Test number that produces the Eitaa "not registered" toast
TEST_UNREGISTERED_PHONE = "9111111111"

wait = WebDriverWait(driver, WAIT_TIMEOUT)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def verify_visible(name, locator):
    """
    Verify that a selector finds a visible element.
    """
    try:
        element = wait.until(
            EC.visibility_of_element_located(locator)
        )

        print(f"✓ {name}")
        return element

    except TimeoutException:
        raise AssertionError(
            f"✗ {name} FAILED\n"
            f"  Locator: {locator}"
        )


def click_element(name, locator):
    """
    Wait for an element to be clickable and click it.
    """
    try:
        element = wait.until(
            EC.element_to_be_clickable(locator)
        )

        element.click()
        print(f"✓ {name} - clicked")
        return element

    except TimeoutException:
        raise AssertionError(
            f"✗ {name} FAILED - element was not clickable\n"
            f"  Locator: {locator}"
        )


# ============================================================
# 1. Verify HAMBURGER_MENU
# ============================================================

print("=" * 70)
print("SECTION 6 - SELENIUM SELECTOR TEST")
print("=" * 70)

print("\n[1] HAMBURGER_MENU")
print("-" * 70)

verify_visible(
    "HAMBURGER_MENU",
    Selectors.HAMBURGER_MENU
)

# ============================================================
# 2. Open hamburger menu
# ============================================================

print("\n[2] Open Hamburger Menu")
print("-" * 70)

click_element(
    "HAMBURGER_MENU",
    Selectors.HAMBURGER_MENU
)

# ============================================================
# 3. Verify CONTACTS_BUTTON
# ============================================================

print("\n[3] CONTACTS_BUTTON")
print("-" * 70)

verify_visible(
    "CONTACTS_BUTTON",
    Selectors.CONTACTS_BUTTON
)

# ============================================================
# 4. Open Contacts
# ============================================================

print("\n[4] Open Contacts")
print("-" * 70)

click_element(
    "CONTACTS_BUTTON",
    Selectors.CONTACTS_BUTTON
)

# ============================================================
# 5. Verify ADD_CONTACT_BUTTON
# ============================================================

print("\n[5] ADD_CONTACT_BUTTON")
print("-" * 70)

verify_visible(
    "ADD_CONTACT_BUTTON",
    Selectors.ADD_CONTACT_BUTTON
)

# ============================================================
# 6. Open Add Contact dialog
# ============================================================

print("\n[6] Open Add Contact Dialog")
print("-" * 70)

click_element(
    "ADD_CONTACT_BUTTON",
    Selectors.ADD_CONTACT_BUTTON
)

# ============================================================
# 7. Verify FIRST_NAME
# ============================================================

print("\n[7] FIRST_NAME")
print("-" * 70)

first_name = verify_visible(
    "FIRST_NAME",
    Selectors.FIRST_NAME
)

# ============================================================
# 8. Verify PHONE
# ============================================================

print("\n[8] PHONE")
print("-" * 70)

phone = verify_visible(
    "PHONE",
    Selectors.PHONE
)

# ============================================================
# 9. Verify SAVE_BUTTON
# ============================================================

print("\n[9] SAVE_BUTTON")
print("-" * 70)

verify_visible(
    "SAVE_BUTTON",
    Selectors.SAVE_BUTTON
)

# ============================================================
# 10. Verify CLOSE_CONTACT_WINDOW
# ============================================================

print("\n[10] CLOSE_CONTACT_WINDOW")
print("-" * 70)

verify_visible(
    "CLOSE_CONTACT_WINDOW",
    Selectors.CLOSE_CONTACT_WINDOW
)

# ============================================================
# 11. Enter test phone number
# ============================================================

print("\n[11] Enter Test Phone Number")
print("-" * 70)

phone.click()
phone.send_keys(TEST_UNREGISTERED_PHONE)

print(
    f"✓ PHONE accepted test number: "
    f"{TEST_UNREGISTERED_PHONE}"
)

# ============================================================
# 12. Enter test first name
# ============================================================

print("\n[12] Enter Test First Name")
print("-" * 70)

first_name.click()
first_name.send_keys("SelectorTest")

print("✓ FIRST_NAME accepted test text")

# ============================================================
# 13. Click SAVE
# ============================================================

print("\n[13] SAVE_BUTTON")
print("-" * 70)

click_element(
    "SAVE_BUTTON",
    Selectors.SAVE_BUTTON
)

# ============================================================
# 14. Detect transient UNREGISTERED_TOAST
#
# IMPORTANT:
# The toast exists only briefly, so we poll the DOM
# frequently instead of using a long normal WebDriverWait.
# ============================================================

print("\n[14] UNREGISTERED_TOAST")
print("-" * 70)

toast_found = False

toast_locator = Selectors.UNREGISTERED_TOAST

toast_deadline = time.monotonic() + 5

while time.monotonic() < toast_deadline:

    try:
        toast_elements = driver.find_elements(
            *toast_locator
        )

        for toast in toast_elements:

            if toast.is_displayed():

                toast_found = True

                print("✓ UNREGISTERED_TOAST appeared")

                try:
                    print(
                        f"  Toast text: "
                        f"{toast.text!r}"
                    )
                except Exception:
                    pass

                break

        if toast_found:
            break

    except Exception:
        pass

    time.sleep(0.05)

if not toast_found:
    raise AssertionError(
        "✗ UNREGISTERED_TOAST FAILED\n"
        "The toast did not appear within the 5-second "
        "monitoring window."
    )

# ============================================================
# 15. CLOSE TOAST
#
# The same CLOSE_CONTACT_WINDOW selector is used.
# The first click dismisses the toast.
# ============================================================

print("\n[15] Close Toast")
print("-" * 70)

click_element(
    "CLOSE_CONTACT_WINDOW - first click (dismiss toast)",
    Selectors.CLOSE_CONTACT_WINDOW
)

print("✓ Toast dismissed")

# ------------------------------------------------------------
# Give Eitaa a very short moment to update the UI.
# ------------------------------------------------------------

time.sleep(0.2)

# ============================================================
# 16. CLOSE ADD CONTACT WINDOW
#
# The same selector must be clicked a second time.
# ============================================================

print("\n[16] Close Add Contact Window")
print("-" * 70)

click_element(
    "CLOSE_CONTACT_WINDOW - second click (close window)",
    Selectors.CLOSE_CONTACT_WINDOW
)

print("✓ Add Contact window close action executed")

# ============================================================
# 17. Verify Add Contact window is actually closed
# ============================================================

print("\n[17] Verify Add Contact Window Closed")
print("-" * 70)

try:
    WebDriverWait(driver, 5).until(
        EC.invisibility_of_element_located(
            Selectors.CLOSE_CONTACT_WINDOW
        )
    )

    print("✓ Add Contact window is closed")

except TimeoutException:
    raise AssertionError(
        "✗ Add Contact window is still visible after "
        "the second CLOSE_CONTACT_WINDOW click."
    )

# ============================================================
# 18. Final result
# ============================================================

print("\n" + "=" * 70)
print("SECTION 6 TEST PASSED")
print("=" * 70)

print("""
All Section 6 Selenium selectors were successfully verified:

✓ HAMBURGER_MENU
✓ CONTACTS_BUTTON
✓ ADD_CONTACT_BUTTON
✓ FIRST_NAME
✓ PHONE
✓ SAVE_BUTTON
✓ UNREGISTERED_TOAST
✓ CLOSE_CONTACT_WINDOW

The transient UNREGISTERED_TOAST was detected successfully.

Cleanup completed:

✓ First CLOSE_CONTACT_WINDOW click dismissed the toast
✓ Second CLOSE_CONTACT_WINDOW click closed the Add Contact window
✓ Add Contact window disappearance was verified
""")


In [10]:
# ============================================================
# 6- Reliable waits
# ============================================================

# Use explicit waits instead of time.sleep().
#
# The purpose of this section is to provide reusable,
# reliable synchronization helpers for the Selenium code
# used in later sections.
#
# These helpers are designed for situations such as:
#   - waiting for Chrome/Eitaa elements to appear
#   - waiting for an element to become visible
#   - waiting for an element to become clickable
#   - waiting for an element/popup to disappear
#   - waiting for a specific Selenium condition
#   - handling temporary/stale DOM elements
#
# ============================================================

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    StaleElementReferenceException,
    NoSuchElementException,
    ElementClickInterceptedException,
)

# ------------------------------------------------------------
# General configuration
# ------------------------------------------------------------

WAIT_SECONDS = 30


# ------------------------------------------------------------
# 1. Create a WebDriverWait object
# ------------------------------------------------------------

def get_wait(driver, timeout=WAIT_SECONDS):
    """
    Return a WebDriverWait instance.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    timeout : int | float
        Maximum number of seconds to wait.

    Returns
    -------
    WebDriverWait
    """
    return WebDriverWait(
        driver,
        timeout,
        poll_frequency=0.2,
        ignored_exceptions=(
            NoSuchElementException,
            StaleElementReferenceException,
        ),
    )


# ------------------------------------------------------------
# 2. Wait until an element is visible
# ------------------------------------------------------------

def wait_visible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present and visible.

    Returns
    -------
    WebElement
    """
    return get_wait(driver, timeout).until(
        EC.visibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 3. Wait until an element is clickable
# ------------------------------------------------------------

def wait_clickable(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is visible and enabled/clickable.

    This should normally be used before clicking buttons,
    links, menu items, etc.
    """
    return get_wait(driver, timeout).until(
        EC.element_to_be_clickable(locator)
    )


# ------------------------------------------------------------
# 4. Wait until an element disappears
# ------------------------------------------------------------

def wait_invisible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is no longer visible.

    This is particularly useful for:
        - closing popups
        - waiting for dialogs to disappear
        - waiting for loading overlays to disappear

    Note:
        Selenium's invisibility_of_element_located() also
        succeeds if the element is removed from the DOM.
    """
    return get_wait(driver, timeout).until(
        EC.invisibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 5. Wait for a custom condition
# ------------------------------------------------------------

def wait_until(
        driver,
        condition,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a custom Selenium condition returns a truthy
    value.

    The condition receives the driver.

    Example:

        wait_until(
            driver,
            lambda d: some_condition(d)
        )

    This is useful when Selenium's built-in expected
    conditions are not sufficient.
    """
    return get_wait(driver, timeout).until(condition)


# ------------------------------------------------------------
# 6. Safe click helper
# ------------------------------------------------------------

def wait_and_click(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is clickable and then click it.

    Returns
    -------
    WebElement
        The element that was clicked.
    """
    element = wait_clickable(
        driver,
        locator,
        timeout,
    )

    element.click()

    return element


# ------------------------------------------------------------
# 7. Wait for a popup to close
# ------------------------------------------------------------

def wait_popup_closed(
        driver,
        popup_locator,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a popup/dialog is no longer visible.

    This is a convenience wrapper around
    wait_invisible().

    It is intended for situations such as the Eitaa
    Add Contact popup.
    """
    return wait_invisible(
        driver,
        popup_locator,
        timeout,
    )


# ------------------------------------------------------------
# 8. Wait for an element and then enter text
# ------------------------------------------------------------

def wait_and_type(
        driver,
        locator,
        text,
        timeout=WAIT_SECONDS,
        clear_first=True,
):
    """
    Wait for an input field to become visible and type text.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    locator : tuple
        Selenium locator, for example:
            (By.CSS_SELECTOR, "...")

    text : str
        Text to enter.

    timeout : int | float
        Maximum number of seconds to wait.

    clear_first : bool
        If True, clear the input before typing.

    Returns
    -------
    WebElement
        The input element.
    """
    element = wait_visible(
        driver,
        locator,
        timeout,
    )

    if clear_first:
        element.clear()

    element.send_keys(text)

    return element

In [11]:

# ============================================================
# 7- Add Contact to Eitaa
# ============================================================

from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException,
)
from selenium.webdriver.common.keys import Keys

UNREGISTERED_MESSAGE = (
    "The person with this phone number is not registered on Eitaa yet."
)


# ------------------------------------------------------------
# 7-1 Detect the unregistered-phone toast
# ------------------------------------------------------------

def _get_unregistered_result(driver):
    """Return True if the unregistered-phone toast is visible."""

    try:
        for toast in driver.find_elements(
                *Selectors.UNREGISTERED_TOAST
        ):
            if (
                    toast.is_displayed()
                    and UNREGISTERED_MESSAGE in toast.text.strip()
            ):
                return True

    except WebDriverException:
        return False

    return False


# ------------------------------------------------------------
# 7-2 Wait for the result of the single Save attempt
# ------------------------------------------------------------

def _wait_for_add_contact_result(driver):
    """
    Return:
        'skipped' when the unregistered toast appears.
        'success' when the contact window closes.

    Do not click Save again if the result is uncertain.
    """

    def result_condition(d):
        # Check for the unregistered result first.
        if _get_unregistered_result(d):
            return "skipped"

        # A missing/hidden close button indicates that the
        # Add Contact window has closed, but is not independent
        # proof that the contact was saved.
        close_buttons = d.find_elements(
            *Selectors.CLOSE_CONTACT_WINDOW
        )

        if not any(button.is_displayed() for button in close_buttons):
            return "success"

        return False

    return wait_until(
        driver,
        result_condition,
        timeout=WAIT_SECONDS,
    )


# ------------------------------------------------------------
# 7-3 Close an unregistered contact's toast and window
# ------------------------------------------------------------

def _close_unregistered_contact_window(driver):
    """
    Follow the observed Eitaa UI behavior:

    First click:
        Dismiss the toast.

    Second click:
        Close the Add Contact window.

    Never click the second time until the toast is gone.
    """

    # Step 1: Confirm that the toast is currently visible.
    if not _get_unregistered_result(driver):
        raise RuntimeError(
            "Cannot dismiss the unregistered result: "
            "the expected toast is not visible."
        )

    # Step 2: First click dismisses the toast.
    wait_and_click(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
        timeout=WAIT_SECONDS,
    )

    # Step 3: Wait for the toast to disappear.
    wait_until(
        driver,
        lambda d: not _get_unregistered_result(d),
        timeout=WAIT_SECONDS,
    )

    # Step 4: The contact window should still be open.
    # Re-find the button because the DOM may have changed.
    close_button = wait_clickable(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
        timeout=WAIT_SECONDS,
    )

    # Step 5: Second click closes the contact window.
    close_button.click()

    # Step 6: Wait until its close button is no longer visible.
    wait_until(
        driver,
        lambda d: not any(
            button.is_displayed()
            for button in d.find_elements(
                *Selectors.CLOSE_CONTACT_WINDOW
            )
        ),
        timeout=WAIT_SECONDS,
    )


# ------------------------------------------------------------
# 7-4 Enter the phone number
# ------------------------------------------------------------

def _enter_phone_number(driver, phone_normalized):
    """Clear the phone field, verify it is empty, then enter 98 + number."""

    phone_number = str(phone_normalized).strip()

    if not phone_number:
        raise ValueError("Phone number cannot be empty.")

    if phone_number.startswith("98"):
        raise ValueError(
            "phone_normalized must not contain country code 98."
        )

    phone_field = wait_clickable(
        driver,
        Selectors.PHONE,
    )

    phone_field.click()
    phone_field.send_keys(Keys.CONTROL, "a")
    phone_field.send_keys(Keys.BACKSPACE)

    def phone_field_is_cleared(d):
        element = d.find_element(*Selectors.PHONE)
        value = (element.get_attribute("value") or "").strip()
        return value in ("", "+")

    wait_until(
        driver,
        phone_field_is_cleared,
        timeout=5,
    )

    # Re-find the element in case React/Eitaa replaced it.
    phone_field = wait_clickable(
        driver,
        Selectors.PHONE,
    )
    phone_field.send_keys("98" + phone_number)


# ------------------------------------------------------------
# 7-5 Main add_contact function
# ------------------------------------------------------------

def add_contact(full_name, phone_normalized):
    """
    Process one contact in one addition attempt.

    Returns:
        'success' - the contact window closed without an
                    unregistered-phone result.
        'skipped' - Eitaa reported an unregistered phone number.

    Unexpected errors propagate to Section 8. Save is never
    automatically retried.
    """

    # Open the form.
    wait_and_click(
        driver,
        Selectors.ADD_CONTACT_BUTTON,
    )

    # Enter the name and phone number.
    wait_and_type(
        driver,
        Selectors.FIRST_NAME,
        str(full_name),
    )

    _enter_phone_number(
        driver,
        phone_normalized,
    )

    # Submit exactly once.
    wait_and_click(
        driver,
        Selectors.SAVE_BUTTON,
    )

    # Wait for Eitaa's response.
    result = _wait_for_add_contact_result(driver)

    if result == "success":
        return "success"

    if result == "skipped":
        _close_unregistered_contact_window(driver)
        return "skipped"

    raise RuntimeError(
        f"Unexpected add_contact result: {result!r}"
    )


In [24]:
# ============================================================
# Section 8: Main Processing Loop
# ============================================================

success_count = 0
skipped_count = 0

print("=" * 60)
print("Section 8: Main Processing Loop")
print("=" * 60)

for _, row in df.iterrows():
    contact_id = row["contact_id"]
    full_name = row["full_name"]
    phone_normalized = row["phone"]

    result = add_contact(
        full_name=full_name,
        phone_normalized=phone_normalized
    )

    if result == "success":
        success_count += 1

    elif result == "skipped":
        skipped_count += 1

    else:
        # An unexpected return value indicates a programming error.
        raise ValueError(
            f"Contact ID {contact_id}: "
            f"Unexpected return value {result!r}"
        )

# ------------------------------------------------------------
# Final Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Section 8: Processing Summary")
print("=" * 60)

print(f"Total contacts: {len(df)}")
print(f"Success:        {success_count}")
print(f"Skipped:        {skipped_count}")

print("=" * 60)
print("Section 8 completed.")

Section 8: Main Processing Loop

Section 8: Processing Summary
Total contacts: 500
Success:        66
Skipped:        434
Section 8 completed.
